In [6]:
# --- Rider object ---
class Rider:
    def __init__(self, name, phone, flight, time_slots):
        self.name = name
        self.phone = phone
        self.flight = flight
        self.time_slots = time_slots
        self.assigned_slot = None

    def __repr__(self):
        return f"Rider({self.name}, slots={self.time_slots})"


# --- Queue Node ---
class RiderNode:
    def __init__(self, rider):
        self.rider = rider
        self.next = None


# --- Queue ---
class RiderQueue:
    def __init__(self):
        self.head = None
        self.tail = None
        self.size = 0

    def is_empty(self):
        return self.size == 0

    def enqueue(self, rider):
        new_node = RiderNode(rider)
        if self.is_empty():
            self.head = new_node
        else:
            self.tail.next = new_node
        self.tail = new_node
        self.size += 1

    def dequeue(self):
        if self.is_empty():
            return None
        node = self.head
        self.head = node.next
        if self.head is None:
            self.tail = None
        self.size -= 1
        return node.rider

    def remove_rider(self, rider):
        prev = None
        curr = self.head
        while curr:
            if curr.rider == rider:
                if prev is None:
                    self.head = curr.next
                    if self.head is None:
                        self.tail = None
                else:
                    prev.next = curr.next
                    if curr == self.tail:
                        self.tail = prev
                self.size -= 1
                return True
            prev = curr
            curr = curr.next
        return False

    def to_list(self):
        res = []
        curr = self.head
        while curr:
            res.append(curr.rider)
            curr = curr.next
        return res


# --- Time validation helper ---
def is_valid_time(ts):
    minutes = ts % 100
    return (minutes % 15 == 0) or (minutes == 0)


# --- BST Node for time slots ---
class TimeSlotNode:
    def __init__(self, time_slot):
        self.time_slot = time_slot
        self.left = None
        self.right = None
        self.queue = RiderQueue()

    def insert_rider(self, time_slot, rider):
        if time_slot == self.time_slot:
            self.queue.enqueue(rider)
        elif time_slot < self.time_slot:
            if not self.left:
                self.left = TimeSlotNode(time_slot)
            self.left.insert_rider(time_slot, rider)
        else:
            if not self.right:
                self.right = TimeSlotNode(time_slot)
            self.right.insert_rider(time_slot, rider)


# --- BST for all rider queues ---
class RideMatchBST:
    def __init__(self):
        self.root = None

    def _find_node(self, node, time_slot):
        if node is None:
            return None
        if time_slot == node.time_slot:
            return node
        if time_slot < node.time_slot:
            return self._find_node(node.left, time_slot)
        return self._find_node(node.right, time_slot)

    def find_node(self, time_slot):
        return self._find_node(self.root, time_slot)

    def choose_best_time_slot(self, rider):
        non_empty = []
        empty = []

        for ts in rider.time_slots:
            node = self.find_node(ts)
            if node is None or node.queue.size == 0:
                empty.append(ts)
            else:
                non_empty.append((ts, node.queue.size))

        if non_empty:
            non_empty.sort(key=lambda x: x[1])
            return non_empty[0][0]

        return min(empty)

    def insert_rider(self, rider):
        valid = [ts for ts in rider.time_slots if is_valid_time(ts)]
        if not valid:
            print(f"Skipping {rider.name}: no valid 15-minute time slots.")
            return

        rider.time_slots = valid
        best = self.choose_best_time_slot(rider)
        rider.assigned_slot = best   

        if not self.root:
            self.root = TimeSlotNode(best)
            self.root.queue.enqueue(rider)
        else:
            self.root.insert_rider(best, rider)

    def remove_rider_by_phone(self, phone):
        """Remove ALL riders with this phone across all time slots."""
        removed_any = False

        for node in self.get_time_slot_nodes():
            riders = node.queue.to_list()
            for r in riders:
                if r.phone == phone:
                    node.queue.remove_rider(r)
                    removed_any = True

        return removed_any

    def _inorder(self, node, result):
        if not node:
            return
        self._inorder(node.left, result)
        result.append(node)
        self._inorder(node.right, result)

    def get_time_slot_nodes(self):
        res = []
        self._inorder(self.root, res)
        return res

    def group_riders(self, car_capacity=4):
        groups_by_slot = {}

        def walk(node):
            if not node:
                return
            walk(node.left)

            groups = []
            group = []
            curr = node.queue.head
            while curr:
                group.append(curr.rider)
                if len(group) == car_capacity:
                    groups.append(group)
                    group = []
                curr = curr.next
            if group:
                groups.append(group)

            groups_by_slot[node.time_slot] = groups
            walk(node.right)

        walk(self.root)
        return groups_by_slot





#      UI SECTION      #


import ipywidgets as widgets
from IPython.display import display, clear_output

# Time parsing/formatting helpers
def parse_time(t):
    t = t.strip().lower().replace(" ", "")
    if not t:
        raise ValueError("empty")
    ampm = None
    if t.endswith("am") or t.endswith("pm"):
        ampm = t[-2:]
        t = t[:-2]
    t = t.replace(":", "")
    if not t.isdigit():
        raise ValueError(f"invalid: {t}")
    if len(t) <= 2:
        h = int(t)
        m = 0
    elif len(t) == 3:
        h = int(t[0])
        m = int(t[1:])
    elif len(t) == 4:
        h = int(t[:2])
        m = int(t[2:])
    else:
        raise ValueError("bad format")
    if ampm == "pm" and h != 12:
        h += 12
    if ampm == "am" and h == 12:
        h = 0
    return h * 100 + m

def parse_times(txt):
    return [parse_time(p) for p in txt.split(",") if p.strip()]

def fmt_time(ts):
    return f"{ts//100:02d}:{ts%100:02d}"


# Widgets
name_in  = widgets.Text(placeholder="name")
phone_in = widgets.Text(placeholder="phone")
flight_in = widgets.Text(placeholder="flight")
time_in  = widgets.Text(placeholder="times: 8am, 10:30")

add_btn   = widgets.Button(description="add rider")
show_btn  = widgets.Button(description="show slots")
group_btn = widgets.Button(description="group riders")
cancel_phone_in = widgets.Text(placeholder="phone to cancel")
cancel_btn = widgets.Button(description="cancel by phone")

out = widgets.Output()

def show_state():
    print("Current time slots:")
    for node in bst.get_time_slot_nodes():
        names = [r.name for r in node.queue.to_list()]
        print(f"{fmt_time(node.time_slot)} → {names}")

@out.capture(clear_output=True)
def add_clicked(_):
    try:
        r = Rider(
            name_in.value.strip(),
            phone_in.value.strip(),
            flight_in.value.strip(),
            parse_times(time_in.value.strip())
        )
    except Exception as e:
        print("Error:", e)
        return
    bst.insert_rider(r)
    print("Added:", r)
    show_state()

@out.capture(clear_output=True)
def show_clicked(_):
    show_state()

@out.capture(clear_output=True)
def group_clicked(_):
    groups = bst.group_riders()
    for ts, cars in sorted(groups.items()):
        print(fmt_time(ts))
        for i, car in enumerate(cars, 1):
            print(f"  Car {i}: {[r.name for r in car]}")

@out.capture(clear_output=True)
def cancel_clicked(_):
    phone = cancel_phone_in.value.strip()
    if not phone:
        print("Enter phone.")
        return

    removed = bst.remove_rider_by_phone(phone)
    if removed:
        print(f"Removed all riders with phone {phone}.")
    else:
        print("No matching riders.")
    show_state()


add_btn.on_click(add_clicked)
show_btn.on_click(show_clicked)
group_btn.on_click(group_clicked)
cancel_btn.on_click(cancel_clicked)

display(name_in, phone_in, flight_in, time_in,
        add_btn, show_btn, group_btn,
        cancel_phone_in, cancel_btn, out)

with out:
    clear_output()
    show_state()


Text(value='', placeholder='name')

Text(value='', placeholder='phone')

Text(value='', placeholder='flight')

Text(value='', placeholder='times: 8am, 10:30')

Button(description='add rider', style=ButtonStyle())

Button(description='show slots', style=ButtonStyle())

Button(description='group riders', style=ButtonStyle())

Text(value='', placeholder='phone to cancel')

Button(description='cancel by phone', style=ButtonStyle())

Output()

In [5]:
#Run this chunk to clear the tree and UI after running the main code chunk above
bst = RideMatchBST()
print("BST reset. Current slots:")
print(bst.get_time_slot_nodes())


BST reset. Current slots:
[]
